# UTS Exploratory Data Analysis (EDA)
**Nama:** Andi Liani  
**NIM:** 1003240053  
**Mata Kuliah:** Machine Learning (IKD-3204)

## SOAL 1 — AUDIT AWAL DATASET

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('bank_credit_data.csv')

print("--- Audit Awal Dataset ---")

# 1. Dimensi dataset
print(f"Dimensi dataset: {df.shape}")

# 2. Penggunaan memori
memory_usage = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Total penggunaan memori: {memory_usage:.2f} MB")

# 3. Tipe data
print("\nTipe Data Setiap Kolom:")
print(df.dtypes)

# 4. Data duplikat
duplicate_count = df.duplicated().sum()
duplicate_pct = (duplicate_count / len(df)) * 100
print(f"\nJumlah data duplikat: {duplicate_count} ({duplicate_pct:.2f}%)")

# 5. Missing values
missing_values = df.isnull().sum()
missing_pct = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Kolom': missing_values.index,
    'Jumlah Missing': missing_values.values,
    'Persentase (%)': missing_pct.values
})
print("\nTabel Ringkasan Missing Values:")
print(missing_df[missing_df['Jumlah Missing'] > 0].to_string(index=False))

## SOAL 2 — DATA CLEANING

In [ ]:
print(f"Jumlah baris sebelum cleaning: {len(df)}")

# 1. Hapus duplikat
df = df.drop_duplicates()
print(f"Jumlah baris sesudah hapus duplikat: {len(df)}")

# 2. Standardisasi kota_domisili
def standardize_city(city):
    city = city.upper()
    if city in ['JKT', 'JAKARTA']: return 'JAKARTA'
    if city in ['BDG', 'BANDUNG']: return 'BANDUNG'
    if city in ['SUB', 'SURABAYA']: return 'SURABAYA'
    if city in ['MEDAN']: return 'MEDAN'
    return city

df['kota_domisili'] = df['kota_domisili'].apply(standardize_city)

# 3. Imputasi median untuk kolom numerik spesifik
numeric_cols = ['skor_kredit', 'utilisasi_kartu', 'jumlah_kartu_kredit', 'frekuensi_login_bulanan']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# 4. Imputasi pendapatan_pasangan dengan 0
df['pendapatan_pasangan'] = df['pendapatan_pasangan'].fillna(0)

# 5. Verifikasi
missing_after = df.isnull().sum().sum()
print(f"\nTotal missing values setelah cleaning: {missing_after}")

## SOAL 3 — FEATURE ENGINEERING

In [ ]:
# 1. rasio_cicilan_pendapatan
df['rasio_cicilan_pendapatan'] = (df['cicilan_bulanan'] / df['pendapatan_bulanan']).clip(0, 1)

# 2. rasio_utang_aset
df['rasio_utang_aset'] = (df['total_utang'] / df['total_aset']).clip(0, 5)

# 3. pendapatan_total
df['pendapatan_total'] = df['pendapatan_bulanan'] + df['pendapatan_pasangan']

# 4. Nilai Tambah: Fitur saldo_terpakai
df['saldo_terpakai'] = df['utilisasi_kartu'] * df['limit_kredit_total']

print("--- Sampel 5 Baris Fitur Baru ---")
print(df[['rasio_cicilan_pendapatan', 'rasio_utang_aset', 'pendapatan_total', 'saldo_terpakai']].head())

## SOAL 4 — VISUALISASI EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Visualisasi: Class Imbalance
plt.figure(figsize=(8, 5))
sns.countplot(x='gagal_bayar', data=df, hue='gagal_bayar', palette='viridis', legend=False)
plt.title('Distribusi Label Gagal Bayar')
plt.show()

## SOAL 5 — ANALISIS RISIKO KREDIT

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()['gagal_bayar'].sort_values()
print("Korelasi Fitur terhadap Gagal Bayar:")
print(corr)